---
title: Land Ice Applications
authors:
  - name: Ben Smith
  - name: Rohaiz Haris
---

# Land Ice Applications

## Learning Outcomes

- Understand how different ICESat-2 land ice products (ATL14/15, ATL11, ATL06) relate to each other
- Learn the basics of retrieving land ice products using `icepyx`
- Learn how ICESat-2 land ice products can be used to study ice-surface height changes
- Open, plot, and explore gridded and along-track data

### Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [xarray](https://foundations.projectpythia.org/core/xarray/xarray-intro/) | Necessary | ICESat-2 data in this tutorial will appear as xarray dataframes |
| [numpy/matplotlib](https://foundations.projectpythia.org/core/matplotlib) | Necessary | We will do our plotting with matplotlib|
| [mission-overview.ipynb](mission-overview.ipynb) | Necessary | Here is where to go to understand the ICESat-2 mission and its goals |

- **Time to learn**:  30 min.

In [ ]:
# import packages:
import numpy as np                   # Numeric Python
import matplotlib.pyplot as plt      # Plotting routines
import h5py                          # general HDF5 reading/writing library
import xarray as xr
from pyproj import Transformer, CRS  # libraries to allow coordinate transforms
import glob                          # Package to locate files on disk
import os                            # File-level utilities
import re                            # regular expressions for string interpretation
import icepyx as ipx                 # Package to interact with ICESat-2 online resources
import earthaccess
from matplotlib import cm
import matplotlib as mpl
#%matplotlib widget

## What is Land Ice?

Land ice is frozen water that is grounded on land. They are formed through the accumulation of snowfall in regions where annual snowfall exceeds annual snow melt. The accumulation of snow over thousands of years leads to the compression of snow into dense ice. Over 99 percent of Earth's land ice is present in the Antarctic and Greenland Ice Sheets, and the remaining are glaciers located in alpine regions. Land ice heights evolve with time due to ice flow and accumulation/ablation.

## How ICESat-2 measures land ice height

Laser altimeters are valuable instruments for measuring land ice height across large spatial areas. ICESat-2 works by sending out photons that bounce off the ice surface; by recording how long it takes for the photons to travel to the ice and back, the satellite can calculate land ice height. ICESat-2’s 91-day repeat cycle enables the monitoring of both seasonal and annual variations in ice surface height. For more details of the ICESat-2 mission, check out the [mission overview page](mission-overview.ipynb)

**ATL06** provides geolocated land-ice surface heights relative to the WGS84 ellipsoid. It is derived from geolocated photons of the **ATL03** product over short segments (40 m).

**ATL11** provides a time-series of land-ice surface heights derived from the **ATL06** product.

**ATL14** is a spatially continuous gridded product of land-ice surface heights derived from the **ATL11** product. **ATL15** provides gridded surface height-change maps over time intervals as fine as 3 months.


## The Problem

We will be looking at Svalbard in the Norwegian Arctic, focusing on the massive surge from the Austfonna ice cap. This started in 2010, and the ice cap is still adjusting to the rapid loss of ice, so we expect to see large thinning rates in the area affected by the surge. We will use the ICESat-2 ATL15 product for a look at the mass-loss pattern over the last three years.


![Svalbard](./images/feart-08-00156-g001_small.jpg)


"Photo credit: Schuler et. al, Front. Earth Sci., 27 May 2020 | https://doi.org/10.3389/feart.2020.00156"

## ATL15

ATL15 is one of the various ICESat-2 data products. ATL15 provides various resolutions (1 km, 10 km, 20 km, and 40 km) of gridded raster data of height change at 3-month intervals, allowing for visualization of height-change patterns and calculation of integrated active subglacial lake volume change (Smith and others, 2022).

ATL14 is an accompanying high-resolution (100 m) digital elevation model (DEM) that provides spatially continuous gridded data of ice sheet surface height.

ATL14 and ATL15 are higher level products derived from lower level products (ATL11) that allow for simplified volume change calculations. However, the finest temporal resolution is 3 months which may be too coarse for certain applications. ATL14 estimates also degrade where measurements are unavailable. Potential use cases for ATL14 and ATL15 are gridded estimates of height change for ice sheet models or the creation of land ice mass balance estimates

We can use the icepyx library to access ICESat-2 data. For a deeper dive into icepyx, check out their [documentation](https://icepyx.readthedocs.io/en/latest/index.html). We first query for the spatial region and the time period that we are interested in


In [ ]:
region_a = ipx.Query('ATL15',[10, 79, 30, 81],['2019-01-01','2023-01-01'])

We can visualize this spatial extent also.

In [ ]:
region_a.visualize_spatial_extent()


Let us now place an order for the granules that are available over our spatial region and time period and download the files. This following cell will prompt you for your earthdata account username and password. Creating an account is free and can be done [here](https://urs.earthdata.nasa.gov/).

In [ ]:
check_fileexist = glob.glob('./data/ATL15*')
if not check_fileexist:
    order = region_a.order_granules(subset=True) 
    files = order.download_granules("./data")
else:
    print("ATL15 files already exist")


ATL15 data comes in NetCDF format. Let's take a look at the structure of these files.

In [ ]:
f_name = glob.glob("./data/ATL15*")
f = xr.open_datatree(f_name[0])
print(f)

We can see that there are many groups in this file. We’ll be plotting the delta_h data variable in this tutorial, here’s what we can learn about from these sources:

* ATL14/15’s [overview page](https://nsidc.org/data/atl15/versions/2): this is likely the ‘quarterly height changes’ described, but let’s dive deeper to be sure
* ATL14/15’s Xarray Dataset imbedded metadata tells us a couple things: delta_h =height change at 1 km (the resolution selected earlier) and height change relative to the datum (Jan 1, 2020) surface
* ATL14/15’s [data dictionary](https://nsidc.org/sites/default/files/documents/technical-reference/icesat2_atl15_data_dict_v002.pdf): delta_h = quarterly height change at 40 km

Ok, since the data is relative to a datum, we have two options:

* Difference individual time slices to subtract out the datum, like so:

    * (time<sub>0</sub> - datum) - (time<sub>1</sub> - datum) = time<sub>0</sub> - datum - time<sub>1</sub> + datum = time<sub>0</sub> - time<sub>1</sub>

* Subtract out the datum directly. The datum is the complementary dataset high-resolution DEM surface contained in tha accompanying dataset ATL14.

In this tutorial we’ll use the first method to look at how the ice surface height has evolved in time.

Lets look at the structure of the file

In [ ]:
ds = xr.open_dataset(f_name[0], group='delta_h')
ds

Lets look at the difference in ice-surface height over the first quarter and plot them

In [ ]:
fig, ax = plt.subplots(figsize=(4,5), tight_layout=True)
dhdt = ds['delta_h'][1,:,:] - ds['delta_h'][0,:,:]
extent=np.array([np.min(ds['x'])-500, np.max(ds['x'])+500, np.min(ds['y'])-500, np.max(ds['y'])+500])
cb = ax.imshow(dhdt, origin='lower', cmap='coolwarm_r', extent=extent, aspect="auto", vmin=dhdt.min(), vmax=-dhdt.min())
ax.set_xlabel('Polar Stereographic X (m)'); ax.set_ylabel('Polar Stereographic Y (m)')
plt.colorbar(cb, fraction=0.02, label='height change [m]')
plt.show()

We see the difference in ice surface height for all of Svalbard. Let us zoom in to the feature of interest in Austfonna Glacier.

In [ ]:
xlim = [1e6, 1.1e6]
ylim = [-4.5e5, -3.5e5]

fig, ax = plt.subplots(figsize=(5,4), tight_layout=True)
dhdt = ds['delta_h'][1,:,:] - ds['delta_h'][0,:,:]
extent=np.array([np.min(ds['x'])-500, np.max(ds['x'])+500, np.min(ds['y'])-500, np.max(ds['y'])+500])
cb = ax.imshow(dhdt, origin='lower', cmap='coolwarm_r', extent=extent, aspect="auto", vmin=dhdt.min(), vmax=-dhdt.min())
ax.set_xlabel('Polar Stereographic X (m)'); ax.set_ylabel('Polar Stereographic Y (m)')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.colorbar(cb, fraction=0.02, label='height change [m]')


We can see a clear signal of significant decrease in height at the center of this glacier leading out into the coastline. Now what if we want to see how the ice surface height evolves over a transect? Let's try that.

In [ ]:
### TBC Code

TBC what we are seeing in the profiles

What if we don't want to deal with interpolated data, and would rather see the data collected over an ICESat-2 reference ground track? Enter ATL11

## ATL11

We might be interested to see where these height changes come from. ATL15 is derived from the ATL11 (slope-corrected along-track height change) product, which maps height changes for individual ICESat-2 reference tracks. It contains data for all cycles with along-track data following the Reference Ground Tracks (RGTs) and allows for the easy calculation of along-track height-change through time. However, it may not work well over complex surfaces and its 120m resolution might be too coarse for certain use cases. Potential use cases would be to make large-scale estimates of glacier and ice sheet height change.

We can use icepyx to find out what track has contributed to the height change at any given point, but for the purpose of this tutorial we shall use track 0548 which we know crosses over our region of interest. The icepyx library takes in spatial extent search parameters as latitude and longitude while our ICESat-2 data is in polar stereographic coordinates. As such, we make some coordinate transformations first in order to search for ATL11 data.


In [ ]:
# Prepare coordinate transformations between lat/lon and the ATL15 coordinate system
crs=CRS.from_epsg(3413)
to_xy_crs=Transformer.from_crs(crs.geodetic_crs, crs)
to_geo_crs=Transformer.from_crs(crs, crs.geodetic_crs)


corners_lat, corners_lon=to_geo_crs.transform(np.array(xlim)[[0, 1, 1, 0, 0]], np.array(ylim)[[0, 0, 1, 1, 0]])
latlims=[np.min(corners_lat), np.max(corners_lat)]
lonlims=[np.min(corners_lon), np.max(corners_lon)]

Now that we've made the necessary coordinate transformations, let us make a query for the data we want.

In [ ]:
region_a = ipx.Query('ATL11', [lonlims[0], latlims[0], lonlims[1], latlims[1]], ['2019-01-01','2023-01-01'], \
                          start_time='00:00:00', end_time='23:59:59', tracks = ['0548'])

We can now order and download the data.

In [ ]:
check_fileexist = glob.glob('./data/ATL11*')
if not check_fileexist:
    region_a.avail_granules()
    order = region_a.order_granules(subset=True) 
    files = order.download_granules("./data")
else:
    print("ATL11 files already exist")



We retrieve two files. Based on the naming convention, the 'ATL11_054803' file is the ascending track in the northern mid latitudes and the 'ATL11_054804' file is the descending track in the mid-northern-hemisphere. You can read more about the naming convention in the ATL11 [user guide](https://nsidc.org/sites/default/files/documents/user-guide/atl11-v006-userguide_0.pdf).

ATL11 files are in HDF5 format. We can use the glob package to find the files in the folder we've downloaded and the h5py package to open the ascending track file and look at its structure.

In [ ]:
ATL11_file=glob.glob('./data/ATL11_054803*')[0]
ATL11 = h5py.File(ATL11_file)
list(ATL11.keys())

The ‘pt1’, ’pt2’, and ‘pt3’ groups contain data for the left, middle and right pair tracks. It’s in these tracks that we find the data. Let's take a look at the middle beam, 'pt2'.

In [ ]:
list(ATL11['pt2'].keys())

We are interested in 'h_corr', the corrected ice surface height. 'cycle_number' lists the cycles that are used for this file and are continually being updated as the ICESat-2 mission progresses. We only want the points over our feature of interest so let us subset the file based on our plot limits as follows.

In [ ]:
lat = ATL11['pt2']['latitude'][:]
lon = ATL11['pt2']['longitude'][:]
h_corr = ATL11['pt2']['h_corr'][:,:]
cyc_num = ATL11['pt2']['cycle_number'][:]
h_corr[h_corr==ATL11['pt2']['h_corr'].attrs['_FillValue']]=np.nan

x_atl11,y_atl11 = to_xy_crs.transform(lat, lon)

ind = x_atl11 > xlim[0]
ind &= x_atl11 < xlim[1]
ind &= y_atl11 > ylim[0]
ind &= y_atl11 < ylim[1]

We can now go ahead and plot the 0548 track over our previous ATL15 data to understand the spatial context of our downloaded ATL11 file.

In [ ]:

fig, ax = plt.subplots(figsize=(5,4), tight_layout=True)
ax.imshow(dhdt, origin='lower', cmap='coolwarm_r', extent=extent, aspect="auto", vmin=dhdt.min(), vmax=-dhdt.min())
#plt.gca().set_aspect(1)
plt.plot(x_atl11,y_atl11,'.')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.gca().set_xlabel('Polar Stereographic X (m)')
plt.gca().set_ylabel('Polar Stereographic Y (m)')

plt.tight_layout()

Our ATL11 file crosses over the high signal of height change. Let us see how the height profile along this track evolves with each cycle (essentially as a function of time).

In [ ]:
fig, ax = plt.subplots(figsize=(5,4), tight_layout=True)

norm = mpl.colors.Normalize(vmin=min(cyc_num), vmax=max(cyc_num))
cmap = cm.get_cmap('viridis')


for cycle in cyc_num:
    cax = ax.plot(x_atl11[ind],h_corr[ind,cycle-3],color=cmap(norm(cycle)))
plt.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, label = "Cycle Number")
plt.legend()
plt.gca().set_xlabel('Polar Stereographic X (m)')
plt.gca().set_ylabel('ATL11 Height (m)')

We can see that at the right and left edges of plot, the ice surface height remains mostly constant. However, in the middle (which lies over the region of strong height change signal), we can see the height decrease as the cycle number increases (and time passes).

What if we want to look at the data that went into making ATL11? Let's take a look at ATL06.

## ATL06
ATL11 data are generated by combining ALT06 data collected over the same ground track, for different cycles. We can use CMR to identify data from a specific ground track. ATL06 contains overlapping 40-meter linear segments that are fit to land and land-ice photons. It provides estimated surface heights with cm-level corrections and is a lighter and higher-level data product than the ATL03 geolocated photon product. However, its 40m resolution is too coarse for some applications and it is only designed for single surface returns. Potential use cases include making large-scale repeatable measurements of glaciers and ice sheets.

Let's use icepyx to get the ATL06 files for the 0548 track we have been looking at. To avoid downloading too much data for this tutorial, let's look into the ATL06 files collected over cycle 3.

In [ ]:
region_b = ipx.Query('ATL06', spatial_extent=[lonlims[0], latlims[0], lonlims[1], latlims[1]], date_range=['2019-01-01','2023-01-01'], \
    cycles = ["03"], tracks=['0548'])

In [ ]:
check_fileexist = glob.glob('./data/*ATL06*')
if not check_fileexist:
    region_b.avail_granules()
    order = region_b.order_granules(subset=True)
    files = order.download_granules("./data")
else:
    print("ATL06 files already exist")


Now that we have our ATL06 files, let's take a look at the structure of one of our files.

In [ ]:
ATL06_file=glob.glob('./data/*ATL06*0303*')[0]
ATL06 = h5py.File(ATL06_file)
list(ATL06.keys())

The 'gtxx' tracks refer to the ground tracks for the six beams on ICESat-2. Each ground track is numbered according to the pattern of tracks on the ground from left to right (GT1L, GT1R, GT2L, GT2R, GT3L, GT3R).  The labeling was chosen such that the beam names do not change when the observatory orientation changes. Let us take a look at the beam data for GT2R, which resides in the 'land_ice_segments' group.


In [ ]:
list(ATL06['gt2r']['land_ice_segments'].keys())

We are interested in 'h_li', the land ice height. Let's plot the ground tracks over our ATL15 data.

In [ ]:
lat = ATL06['gt2r']['land_ice_segments']['latitude'][:]
lon = ATL06['gt2r']['land_ice_segments']['longitude'][:]
h_li = ATL06['gt2r']['land_ice_segments']['h_li'][:]
h_li[h_li==ATL06['gt2r']['land_ice_segments']['h_li'].attrs['_FillValue']]=np.nan

x_atl06,y_atl06 = to_xy_crs.transform(lat, lon)

In [ ]:
fig, ax = plt.subplots(figsize=(5,4), tight_layout=True)
ax.imshow(dhdt, origin='lower', cmap='coolwarm_r', extent=extent, aspect="auto", vmin=dhdt.min(), vmax=-dhdt.min())
#plt.gca().set_aspect(1)
plt.plot(x_atl06,y_atl06,'.')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.gca().set_xlabel('Polar Stereographic X (m)')
plt.gca().set_ylabel('Polar Stereographic Y (m)')

plt.tight_layout()

We can now plot the ATL06 data over our ATL11 data for the third cycle and see how they compare.

In [ ]:
fig, ax = plt.subplots(figsize=(5,4), tight_layout=True)

ax.plot(x_atl11[ind],h_corr[ind,0], label="ATL11")
ax.plot(x_atl06,h_li, label = "ATL06")
plt.legend()
plt.gca().set_xlabel('Polar Stereographic X (m)')
plt.gca().set_ylabel('Ice Surface height (m)')

What gives? It's likely that some of the ATL06 data is bad. Fortunately, ATL06 comes with a quality flag (atl06_quality_summary) that identifies segments that are likely good (zero means no problems). Let's filter out the bad data and try this again with the good data.

In [ ]:
fig, ax = plt.subplots(figsize=(5,4), tight_layout=True)

good=(ATL06['gt2r']['land_ice_segments']['atl06_quality_summary'][:]==0)
if np.any(good):
        plt.plot(x_atl06[good],h_li[good],'.', label = "ATL06")
plt.plot(x_atl11[ind],h_corr[ind,0], label = "ATL11")
plt.legend()
plt.gca().set_xlabel('Polar Stereographic X (m)')
plt.gca().set_ylabel('Ice Surface Height (m)')


Great success! We see that our ATL06 data is colocated with the ATL11 data now.

## Conclusion

We have briefly explored how to use different ICESat-2 land ice products to study ice mass loss in Svalbard and how to interact with these datasets. ICESat-2 land ice products are also used widely for studies of surface elevation, elevation change, and gridded land-ice analysis products.

The references below provide a starting point for understanding the products used in this workflow and for finding related applications.

## References

- [Smith et al. (2019), *Land ice height-retrieval algorithm for NASA’s ICESat-2 photon-counting laser altimeter*](https://doi.org/10.1016/j.rse.2019.111352)
- [Smith et al. (2022), *ATLAS/ICESat-2 L3B Gridded Antarctic and Arctic Land Ice Height Change, Version 2*](https://doi.org/10.5067/ATLAS/ATL15.002)
- [ATL15 Data Dictionary (V01)](https://nsidc.org/data/documentation/atl15-data-dictionary-v01)
- [CryoCloud: IS2-ATL15 Surface-Height Anomalies](https://book.cryointhecloud.com/is2-atl15-surface-height-anomalies)